1. Gerekli Kütüphanelerin İçe Aktarılması

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Grafik stili
plt.style.use('ggplot')


2. Veri Hazırlama ve Model Eğitim Fonksiyonu

In [9]:
def prepare_model():
    print("Veri seti yükleniyor ve işleniyor...")
    # Veriyi yükle
    df = pd.read_csv('../dataset/train.csv')

    # Veri ön izleme
    print("\nVeri Seti Önizleme:")
    print(df.head())

    # Özellik mühendisliği
    df['engine_power'] = df['engine'].str.extract(r'(\d+\.?\d*)HP').astype(float)
    df['engine_volume'] = df['engine'].str.extract(r'(\d+\.?\d*)L').astype(float)
    df['cylinders'] = df['engine'].str.extract(r'(\d+) Cylinder').astype(float)
    df['fuel_type'] = df['fuel_type'].str.replace('Gasoline/Mild Electric Hybrid', 'Hybrid')
    df['fuel_type'] = df['fuel_type'].str.replace('Plug-In Electric/Gas', 'Hybrid')
    df['fuel_type'] = df['fuel_type'].str.replace('Gas/Electric Hybrid', 'Hybrid')
    df['age'] = 2023 - df['model_year']
    df['accident'] = df['accident'].apply(lambda x: 1 if 'accident' in str(x) else 0)
    df['clean_title'] = df['clean_title'].apply(lambda x: 1 if str(x) == 'Yes' else 0)
    df['transmission_type'] = df['transmission'].apply(
        lambda x: 'Automatic' if 'A/T' in str(x) else 'Manual' if 'M/T' in str(x) else 'Other')

    # Temizlik
    df = df.dropna(subset=['engine_power', 'engine_volume', 'cylinders'])
    df['clean_title'].fillna(0, inplace=True)

    # Veri dağılımı görselleştirme
    plt.figure(figsize=(15, 10))

    plt.subplot(2, 2, 1)
    sns.histplot(df['price'], bins=30, kde=True)
    plt.title('Fiyat Dağılımı')

    plt.subplot(2, 2, 2)
    sns.scatterplot(x='milage', y='price', data=df)
    plt.title('Kilometre-Fiyat İlişkisi')

    plt.subplot(2, 2, 3)
    sns.boxplot(x='fuel_type', y='price', data=df)
    plt.title('Yakıt Türüne Göre Fiyat Dağılımı')

    plt.subplot(2, 2, 4)
    sns.scatterplot(x='engine_power', y='price', data=df)
    plt.title('Motor Gücü-Fiyat İlişkisi')

    plt.tight_layout()
    plt.savefig('data_distribution.png')
    print("\nVeri dağılım grafikleri 'data_distribution.png' olarak kaydedildi.")
    plt.close()

    return df
prepare_model()

Veri seti yükleniyor ve işleniyor...

Veri Seti Önizleme:
   id          brand              model  model_year  milage      fuel_type  \
0   0           MINI      Cooper S Base        2007  213000       Gasoline   
1   1        Lincoln              LS V8        2002  143250       Gasoline   
2   2      Chevrolet  Silverado 2500 LT        2002  136731  E85 Flex Fuel   
3   3        Genesis   G90 5.0 Ultimate        2017   19500       Gasoline   
4   4  Mercedes-Benz        Metris Base        2021    7388       Gasoline   

                                              engine  \
0       172.0HP 1.6L 4 Cylinder Engine Gasoline Fuel   
1       252.0HP 3.9L 8 Cylinder Engine Gasoline Fuel   
2  320.0HP 5.3L 8 Cylinder Engine Flex Fuel Capab...   
3       420.0HP 5.0L 8 Cylinder Engine Gasoline Fuel   
4       208.0HP 2.0L 4 Cylinder Engine Gasoline Fuel   

                     transmission ext_col int_col  \
0                             A/T  Yellow    Gray   
1                             

,id,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price,engine_power,engine_volume,cylinders,age,transmission_type
0,0,MINI,Cooper S Base,2007,213000,Gasoline,172.0HP 1.6L 4 Cylinder Engine Gasoline Fuel,A/T,Yellow,Gray,0,1,4200,172.0,1.6,4.0,16,Automatic
1,1,Lincoln,LS V8,2002,143250,Gasoline,252.0HP 3.9L 8 Cylinder Engine Gasoline Fuel,A/T,Silver,Beige,1,1,4999,252.0,3.9,8.0,21,Automatic
2,2,Chevrolet,Silverado 2500 LT,2002,136731,E85 Flex Fuel,320.0HP 5.3L 8 Cylinder Engine Flex Fuel Capab...,A/T,Blue,Gray,0,1,13900,320.0,5.3,8.0,21,Automatic
3,3,Genesis,G90 5.0 Ultimate,2017,19500,Gasoline,420.0HP 5.0L 8 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,Black,Black,0,1,45000,420.0,5.0,8.0,6,Other
4,4,Mercedes-Benz,Metris Base,2021,7388,Gasoline,208.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,7-Speed A/T,Black,Beige,0,1,97500,208.0,2.0,4.0,2,Automatic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188527,188527,Chevrolet,Camaro Z28,1999,110000,Gasoline,310.0HP 5.7L 8 Cylinder Engine Gasoline Fuel,A/T,White,Gray,0,1,14500,310.0,5.7,8.0,24,Automatic
188528,188528,Cadillac,Escalade ESV Platinum,2017,49000,Gasoline,420.0HP 6.2L 8 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,White,Beige,0,1,27500,420.0,6.2,8.0,6,Other
188529,188529,Mercedes-Benz,AMG C 43 AMG C 43 4MATIC,2018,28600,Gasoline,385.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,White,Black,1,1,30000,385.0,3.0,6.0,5,Automatic
188530,188530,Mercedes-Benz,AMG GLC 63 Base 4MATIC,2021,13650,Gasoline,469.0HP 4.0L 8 Cylinder Engine Gasoline Fuel,7-Speed A/T,White,Black,0,1,86900,469.0,4.0,8.0,2,Automatic


3. Model Eğitim ve Değerlendirme

In [10]:
def train_and_evaluate_model(df):
    # Kategorik ve sayısal sütunlar
    categorical_cols = ['brand', 'fuel_type', 'transmission_type']
    numerical_cols = ['model_year', 'milage', 'engine_power', 'engine_volume',
                     'cylinders', 'age', 'accident', 'clean_title']

    # Pipeline oluştur
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', numerical_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
        ])

    # Model pipeline'ı
    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', GradientBoostingRegressor(random_state=42))
    ])

    # Eğitim
    X = df[numerical_cols + categorical_cols]
    y = df['price']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("\nModel eğitiliyor...")
    model.fit(X_train, y_train)

    # Model değerlendirme
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print("\nModel Değerlendirme Metrikleri:")
    print(f"Ortalama Mutlak Hata (MAE): {mae:.2f}")
    print(f"Ortalama Kare Hata (MSE): {mse:.2f}")
    print(f"Kök Ortalama Kare Hata (RMSE): {rmse:.2f}")
    print(f"R-Kare (R2) Skoru: {r2:.2f}")

    # Gerçek vs Tahmin grafiği
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=y_test, y=y_pred)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
    plt.xlabel('Gerçek Fiyatlar')
    plt.ylabel('Tahmini Fiyatlar')
    plt.title('Gerçek vs Tahmin Edilen Fiyatlar')
    plt.savefig('actual_vs_predicted.png')
    print("Gerçek-Tahmin grafiği 'actual_vs_predicted.png' olarak kaydedildi.")
    plt.close()

    return model


TypeError: train_and_evaluate_model() missing 1 required positional argument: 'df'

4. Özellik Önem Dereceleri

In [4]:
def plot_feature_importances(model, df):
    try:
        feature_importances = model.named_steps['regressor'].feature_importances_
        cat_encoder = model.named_steps['preprocessor'].named_transformers_['cat']
        cat_features = cat_encoder.get_feature_names_out(['brand', 'fuel_type', 'transmission_type'])
        all_features = ['model_year', 'milage', 'engine_power', 'engine_volume', 'cylinders', 'age', 'accident', 'clean_title'] + list(cat_features)

        importance_df = pd.DataFrame({'Feature': all_features, 'Importance': feature_importances})
        importance_df = importance_df.sort_values('Importance', ascending=False).head(15)

        plt.figure(figsize=(12, 8))
        sns.barplot(x='Importance', y='Feature', data=importance_df)
        plt.title('Özellik Önem Dereceleri (Top 15)')
        plt.tight_layout()
        plt.savefig('feature_importances.png')
        print("Özellik önem dereceleri grafiği 'feature_importances.png' olarak kaydedildi.")
        plt.close()
    except Exception as e:
        print(f"\nÖzellik önem dereceleri çizilemedi: {str(e)}")


5. Modeli Kaydetme veya Yükleme

In [5]:
def get_model():
    model_file = 'car_price_model.pkl'
    if os.path.exists(model_file):
        print("\nÖnceden eğitilmiş model yükleniyor...")
        return joblib.load(model_file)
    else:
        print("\nYeni model eğitiliyor...")
        df = prepare_model()
        model = train_and_evaluate_model(df)
        joblib.dump(model, model_file)
        print(f"Model '{model_file}' olarak kaydedildi.")
        return model


6. Kullanıcı Girişi Alma

In [6]:
def get_user_input():
    print("\n" + "="*50)
    print("Araç Özelliklerini Girin:")
    print("="*50)

    brand = input("\nMarka (Örnek: Toyota, BMW, Ford): ").strip().title()
    model_year = int(input("Model Yılı (Örnek: 2015): "))
    milage = int(input("Kilometre (Örnek: 50000): "))
    fuel_type = input("Yakıt Türü (Gasoline, Diesel, Hybrid, Electric): ").strip().title()
    engine_power = float(input("Motor Gücü (HP) (Örnek: 150): "))
    engine_volume = float(input("Motor Hacmi (L) (Örnek: 2.0): "))
    cylinders = int(input("Silindir Sayısı (Örnek: 4): "))
    transmission_type = input("Şanzıman Türü (Automatic, Manual, Other): ").strip().title()
    accident = input("Kaza Geçmişi Var mı? (Evet/Hayır): ").lower() == 'evet'
    clean_title = input("Temiz Başlık? (Evet/Hayır): ").lower() == 'evet'

    # Yaş hesapla
    current_year = pd.Timestamp.now().year
    age = current_year - model_year

    # Veri sözlüğü oluştur
    input_data = {
        'brand': [brand],
        'model_year': [model_year],
        'milage': [milage],
        'fuel_type': [fuel_type],
        'engine_power': [engine_power],
        'engine_volume': [engine_volume],
        'cylinders': [cylinders],
        'transmission_type': [transmission_type],
        'age': [age],
        'accident': [1 if accident else 0],
        'clean_title': [1 if clean_title else 0]
    }

    return pd.DataFrame(input_data)


7. Ana Uygulama Fonksiyonu

In [7]:
def main():
    print("\n" + "="*50)
    print("Araç Fiyat Tahmini Uygulamasına Hoş Geldiniz!")
    print("="*50)

    # Modeli yükle
    model = get_model()

    while True:
        # Kullanıcı girişi al
        user_data = get_user_input()

        # Tahmin yap
        predicted_price = model.predict(user_data)[0]

        # Sonucu göster
        print("\n" + "="*50)
        print("TAHMİN SONUÇLARI")
        print("="*50)
        print(f"\nGirilen Araç Özellikleri:")
        print(f"- Marka: {user_data['brand'][0]}")
        print(f"- Model Yılı: {user_data['model_year'][0]} (Yaş: {user_data['age'][0]} yıl)")
        print(f"- Kilometre: {user_data['milage'][0]:,} km")
        print(f"- Yakıt Türü: {user_data['fuel_type'][0]}")
        print(f"- Motor Gücü: {user_data['engine_power'][0]} HP")
        print(f"- Motor Hacmi: {user_data['engine_volume'][0]} L")
        print(f"- Silindir Sayısı: {user_data['cylinders'][0]}")
        print(f"- Şanzıman Türü: {user_data['transmission_type'][0]}")
        print(f"- Kaza Geçmişi: {'Evet' if user_data['accident'][0] else 'Hayır'}")
        print(f"- Temiz Başlık: {'Evet' if user_data['clean_title'][0] else 'Hayır'}")

        print("\n" + "-"*50)
        print(f"\nTahmini Araç Fiyatı: ${predicted_price:,.2f}")
        print("-"*50)

        # Devam etmek isteyip istemediğini sor
        another = input("\nBaşka bir tahmin yapmak ister misiniz? (Evet/Hayır): ").lower()
        if another != 'evet':
            print("\nProgram sonlandırılıyor...")
            print("Görselleştirme dosyalarını kontrol etmeyi unutmayın!")
            break

if __name__ == "__main__":
    main()



Araç Fiyat Tahmini Uygulamasına Hoş Geldiniz!

Önceden eğitilmiş model yükleniyor...

Araç Özelliklerini Girin:

TAHMİN SONUÇLARI

Girilen Araç Özellikleri:
- Marka: Mini
- Model Yılı: 2007 (Yaş: 18 yıl)
- Kilometre: 213,000 km
- Yakıt Türü: Gasoline
- Motor Gücü: 172.0 HP
- Motor Hacmi: 1.6 L
- Silindir Sayısı: 4
- Şanzıman Türü: Automatic
- Kaza Geçmişi: Hayır
- Temiz Başlık: Evet

--------------------------------------------------

Tahmini Araç Fiyatı: $8,900.02
--------------------------------------------------

Program sonlandırılıyor...
Görselleştirme dosyalarını kontrol etmeyi unutmayın!
